# Pelanca NNUE - Training Pipeline

Treina uma rede neural NNUE para o engine de xadrez Pelanca Mate v5.

**Requisitos Kaggle:**
- GPU T4 x2 (Accelerator: GPU T4 x2)
- Internet habilitada (Settings > Internet > On)

**Tempo estimado:**
- Download dados: ~30-60 min (10M posicoes)
- Pre-processamento: ~15-20 min
- Treino: ~2-4h (30 epochs)

**Resultado:** arquivo `pelanca.nnue` (~410KB) para copiar em `nn/pelanca.nnue` no projeto Rust.

---
## 1. Setup & Instalacao

In [ ]:
!pip install python-chess datasets h5py tqdm -q

import chess
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import numpy as np
import h5py
import struct
import os
import time
from tqdm.auto import tqdm

print(f'python-chess: {chess.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

---
## 2. Configuracao

In [ ]:
# ============================================================
# CONFIGURACAO - AJUSTE AQUI
# ============================================================

# Download de dados
NUM_POSITIONS = 10_000_000     # 10M posicoes (recomendado para 205K params)
MIN_DEPTH = 10                 # Profundidade minima do Stockfish
HDF5_FILE = '/kaggle/working/positions.h5'

# Treino
EPOCHS = 30
BATCH_SIZE = 16384
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-6

# Output
NNUE_OUTPUT = '/kaggle/working/pelanca.nnue'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Arquitetura NNUE
INPUT_SIZE = 768
FT_SIZE = 256
HIDDEN_SIZE = 32
EVAL_SCALE = 400.0

# Quantizacao (deve coincidir com o Rust)
FT_QUANT_SCALE = 64
HIDDEN_QUANT_SCALE = 64
OUTPUT_QUANT_SCALE = 64

print('Configuracao OK')
print(f'  Posicoes: {NUM_POSITIONS:,}')
print(f'  Ratio dados/params: ~{NUM_POSITIONS // 205000}:1 (recomendado >= 50:1)')

---
## 3. Download de Dados (Lichess + Stockfish evaluations)

Baixa posicoes avaliadas pelo Stockfish direto do HuggingFace.  
**~15-30 min** (vs 2-4h gerando localmente).

In [ ]:
if os.path.exists(HDF5_FILE):
    size_mb = os.path.getsize(HDF5_FILE) / 1024**2
    print(f'Dataset ja existe: {HDF5_FILE} ({size_mb:.0f} MB) - pulando download')
else:
    from datasets import load_dataset
    print(f'Baixando {NUM_POSITIONS:,} posicoes (depth >= {MIN_DEPTH})...')
    ds = load_dataset('Lichess/chess-position-evaluations', split='train', streaming=True)

    all_fens = []
    all_values = []
    total = 0
    seen = set()
    skipped = 0
    t0 = time.time()

    for row in ds:
        if total >= NUM_POSITIONS:
            break

        depth = row.get('depth', 0) or 0
        if depth < MIN_DEPTH:
            skipped += 1
            continue

        fen = row['fen']
        if fen in seen:
            skipped += 1
            continue
        seen.add(fen)

        # Validar posicao
        try:
            board = chess.Board(fen)
            if not board.is_valid():
                skipped += 1
                continue
        except:
            skipped += 1
            continue

        # Valor normalizado com tanh
        cp = row.get('cp')
        mate = row.get('mate')
        if mate is not None:
            value = 1.0 if mate > 0 else -1.0
        elif cp is not None:
            value = float(np.tanh(cp / 400.0))
        else:
            skipped += 1
            continue

        all_fens.append(fen)
        all_values.append(value)
        total += 1

        if total % 10000 == 0:
            rate = total / (time.time() - t0)
            eta = (NUM_POSITIONS - total) / rate / 60
            print(f'\r  {total:,}/{NUM_POSITIONS:,} ({total/NUM_POSITIONS*100:.0f}%) | '
                  f'{rate:.0f}/s | ETA: {eta:.0f}min', end='', flush=True)

    print(f'\n\nTotal: {total:,} posicoes (skipped: {skipped:,})')

    # Shuffle e split train/val (95/5)
    rng = np.random.default_rng(42)
    indices = rng.permutation(total)
    val_size = int(total * 0.05)
    val_idx = indices[:val_size]
    train_idx = indices[val_size:]

    fens_arr = np.array(all_fens, dtype=object)
    values_arr = np.array(all_values, dtype=np.float32)

    print(f'Salvando: {len(train_idx):,} train, {len(val_idx):,} val')
    dt_str = h5py.string_dtype()
    with h5py.File(HDF5_FILE, 'w') as f:
        for name, idx in [('train', train_idx), ('val', val_idx)]:
            g = f.create_group(name)
            g.create_dataset('fens', data=fens_arr[idx], dtype=dt_str)
            g.create_dataset('values', data=values_arr[idx], compression='gzip')
        f.attrs['total'] = total
        f.attrs['format'] = 'v2_full_fen'

    elapsed = time.time() - t0
    size_mb = os.path.getsize(HDF5_FILE) / 1024**2
    print(f'Salvo: {HDF5_FILE} ({size_mb:.0f} MB) em {elapsed/60:.1f} min')

    # Liberar memoria
    del all_fens, all_values, fens_arr, values_arr, seen
    import gc; gc.collect()

---
## 4. Feature Encoding

Deve coincidir **exatamente** com o Rust (`src/engine/eval/nnue.rs`):  
`feature_index = color * 384 + piece_type * 64 + square`

In [ ]:
# Mapeamento: deve bater com PieceKind no Rust
# Pawn=0, Knight=1, Bishop=2, Rook=3, Queen=4, King=5
# White=0, Black=1 | Square: a1=0, b1=1, ..., h8=63

PIECE_TYPE_MAP = {
    chess.PAWN: 0,
    chess.KNIGHT: 1,
    chess.BISHOP: 2,
    chess.ROOK: 3,
    chess.QUEEN: 4,
    chess.KING: 5,
}


def fen_to_features(fen_str):
    """Extrai features NNUE de um FEN. Retorna (stm, white_input[768], black_input[768])."""
    board = chess.Board(fen_str)
    stm = 0 if board.turn == chess.WHITE else 1

    white_input = np.zeros(INPUT_SIZE, dtype=np.float32)
    black_input = np.zeros(INPUT_SIZE, dtype=np.float32)

    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece is None:
            continue

        color = 0 if piece.color == chess.WHITE else 1
        pt = PIECE_TYPE_MAP[piece.piece_type]

        # White perspective: as-is
        w_feat = color * 384 + pt * 64 + sq
        white_input[w_feat] = 1.0

        # Black perspective: flip color + mirror square
        b_color = 1 - color
        b_sq = sq ^ 56  # espelho vertical
        b_feat = b_color * 384 + pt * 64 + b_sq
        black_input[b_feat] = 1.0

    return stm, white_input, black_input


# Teste rapido na posicao inicial
stm, w, b = fen_to_features('rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1')
assert stm == 0
assert w.sum() == 32  # 32 pecas
assert b.sum() == 32
# Posicao inicial e simetrica: white_input == black_input
assert np.array_equal(w, b), 'Perspectiva simetrica falhou!'
print(f'Feature encoding OK (32 pecas, simetria verificada)')

---
## 5. Dataset (representacao ESPARSA - cabe 10M+ em 13GB RAM)

Cada posicao guarda apenas os ~32 indices ativos (u16) em vez de 768 floats.  
**RAM: ~32 indices x 2 bytes x 2 perspectivas x 10M = ~1.3 GB** (vs 15+ GB denso).

In [ ]:
def fen_to_sparse_features(fen_str):
    """Extrai indices ESPARSOS de features de um FEN.
    Retorna (stm, white_indices[], black_indices[]).
    Cada lista tem ~32 elementos (numero de pecas no tabuleiro).
    """
    board = chess.Board(fen_str)
    stm = 0 if board.turn == chess.WHITE else 1

    white_indices = []
    black_indices = []

    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        if piece is None:
            continue

        color = 0 if piece.color == chess.WHITE else 1
        pt = PIECE_TYPE_MAP[piece.piece_type]

        # White perspective: as-is
        w_feat = color * 384 + pt * 64 + sq
        white_indices.append(w_feat)

        # Black perspective: flip color + mirror square
        b_feat = (1 - color) * 384 + pt * 64 + (sq ^ 56)
        black_indices.append(b_feat)

    return stm, white_indices, black_indices


class NnueSparseDataset(Dataset):
    """Dataset ESPARSO: guarda so indices ativos (~1.3 GB para 10M posicoes).

    Na __getitem__, monta o tensor denso 768 on-the-fly (rapido no GPU).
    """

    def __init__(self, h5_path, split='train'):
        with h5py.File(h5_path, 'r') as f:
            g = f[split]
            fens = [x.decode() if isinstance(x, bytes) else x for x in g['fens'][:]]
            self.values = g['values'][:].astype(np.float32)

        self.n = len(fens)
        print(f'[{split}] {self.n:,} posicoes')

        # Pre-computar features ESPARSAS
        print(f'  Extraindo features esparsas...')
        self.stm = np.zeros(self.n, dtype=np.uint8)
        self.white_indices = [None] * self.n
        self.black_indices = [None] * self.n

        for i in tqdm(range(self.n), desc=f'  {split}', leave=False):
            stm, wi, bi = fen_to_sparse_features(fens[i])
            self.stm[i] = stm
            self.white_indices[i] = np.array(wi, dtype=np.int16)
            self.black_indices[i] = np.array(bi, dtype=np.int16)

        del fens  # liberar FENs
        import gc; gc.collect()

        # Calcular uso de memoria
        idx_bytes = sum(w.nbytes + b.nbytes for w, b in zip(self.white_indices, self.black_indices))
        total_mb = (idx_bytes + self.values.nbytes + self.stm.nbytes) / 1e6
        print(f'  Pronto! ({total_mb:.0f} MB em RAM)')

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        stm = self.stm[idx]
        value = self.values[idx]

        # Montar tensores densos a partir dos indices esparsos
        white_input = torch.zeros(INPUT_SIZE, dtype=torch.float32)
        black_input = torch.zeros(INPUT_SIZE, dtype=torch.float32)

        white_input[self.white_indices[idx].astype(np.int64)] = 1.0
        black_input[self.black_indices[idx].astype(np.int64)] = 1.0

        # Ordenar por STM
        if stm == 0:  # White to move
            stm_input, nstm_input = white_input, black_input
            stm_value = value
        else:  # Black to move
            stm_input, nstm_input = black_input, white_input
            stm_value = -value  # inverter perspectiva

        return stm_input, nstm_input, torch.tensor(stm_value, dtype=torch.float32)


# Teste rapido
_stm, _wi, _bi = fen_to_sparse_features('rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1')
assert _stm == 0
assert len(_wi) == 32  # 32 pecas
assert sorted(_wi) == sorted(_bi)  # simetria na posicao inicial
print(f'Dataset esparso OK (32 pecas, ~{32*2*2} bytes/posicao vs {768*4*2} bytes denso)')

---
## 6. Modelo NNUE

In [ ]:
class ClippedReLU(nn.Module):
    def __init__(self, max_val=1.0):
        super().__init__()
        self.max_val = max_val

    def forward(self, x):
        return torch.clamp(x, 0.0, self.max_val)


class PelancaNNUE(nn.Module):
    """
    768 -> FT(768->256, CReLU) x2 perspectivas
    -> Concat(512) -> Hidden(512->32, CReLU)
    -> Output(32->1)

    Output: valor tanh-normalizado [-1, 1] (positivo = bom para STM).
    Conversao para centipawns: atanh(output) * 400
    """

    def __init__(self):
        super().__init__()
        self.ft = nn.Linear(INPUT_SIZE, FT_SIZE)
        self.hidden = nn.Linear(FT_SIZE * 2, HIDDEN_SIZE)
        self.output_layer = nn.Linear(HIDDEN_SIZE, 1)
        self.crelu = ClippedReLU()
        self._init_weights()

    def _init_weights(self):
        nn.init.kaiming_normal_(self.ft.weight, nonlinearity='relu')
        nn.init.zeros_(self.ft.bias)
        nn.init.kaiming_normal_(self.hidden.weight, nonlinearity='relu')
        nn.init.zeros_(self.hidden.bias)
        nn.init.xavier_normal_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)

    def forward(self, stm_input, nstm_input):
        stm_acc = self.crelu(self.ft(stm_input))
        nstm_acc = self.crelu(self.ft(nstm_input))
        combined = torch.cat([stm_acc, nstm_acc], dim=1)
        hidden = self.crelu(self.hidden(combined))
        return self.output_layer(hidden)


def nnue_loss(pred, target):
    """MSE loss. Pred e target ambos em escala tanh [-1, 1]."""
    return F.mse_loss(torch.tanh(pred), target)


# Teste
m = PelancaNNUE()
print(f'Modelo: {sum(p.numel() for p in m.parameters()):,} parametros')
del m

---
## 7. Funcao de Export

Converte pesos PyTorch para formato binario quantizado (i16/i8) que o Rust carrega.

In [ ]:
MAGIC = b'PLNN'
NNUE_VERSION = 1


def export_weights(model, output_path, verbose=True):
    """Exporta pesos do modelo para formato binario do Rust."""
    model.eval()
    m = model.module if hasattr(model, 'module') else model

    with open(output_path, 'wb') as f:
        # Header
        f.write(MAGIC)
        f.write(struct.pack('<I', NNUE_VERSION))
        f.write(struct.pack('<I', INPUT_SIZE))
        f.write(struct.pack('<I', FT_SIZE))
        f.write(struct.pack('<I', HIDDEN_SIZE))

        # Feature Transform (i16)
        ft_w = (m.ft.weight.data.cpu() * FT_QUANT_SCALE).round().clamp(-32767, 32767).to(torch.int16)
        ft_b = (m.ft.bias.data.cpu() * FT_QUANT_SCALE).round().clamp(-32767, 32767).to(torch.int16)
        f.write(ft_w.numpy().tobytes())
        f.write(ft_b.numpy().tobytes())

        # Hidden (i8 weights, i32 biases)
        h_w = (m.hidden.weight.data.cpu() * HIDDEN_QUANT_SCALE).round().clamp(-127, 127).to(torch.int8)
        h_b = (m.hidden.bias.data.cpu() * FT_QUANT_SCALE * HIDDEN_QUANT_SCALE).round().clamp(-2**30, 2**30).to(torch.int32)
        f.write(h_w.numpy().tobytes())
        f.write(h_b.numpy().tobytes())

        # Output (i8 weights, i32 bias)
        o_w = (m.output_layer.weight.data.cpu() * OUTPUT_QUANT_SCALE).round().clamp(-127, 127).to(torch.int8)
        o_b = (m.output_layer.bias.data.cpu() * FT_QUANT_SCALE * HIDDEN_QUANT_SCALE * OUTPUT_QUANT_SCALE).round().clamp(-2**30, 2**30).to(torch.int32)
        f.write(o_w.numpy().tobytes())
        f.write(o_b.numpy().tobytes())

    total_size = os.path.getsize(output_path)
    if verbose:
        print(f'Exportado: {output_path} ({total_size:,} bytes / {total_size/1024:.1f} KB)')


print('Export OK')

---
## 8. Carregar Dataset & DataLoaders

In [ ]:
print('Carregando e pre-computando features...')
print('(isso demora alguns minutos na primeira vez)\n')

train_dataset = NnueHDF5Dataset(HDF5_FILE, split='train', precompute=True)
val_dataset = NnueHDF5Dataset(HDF5_FILE, split='val', precompute=True)

num_workers = min(4, os.cpu_count() or 1)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=num_workers, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=num_workers, pin_memory=True,
)

print(f'\nTrain batches/epoch: {len(train_loader)}')
print(f'Val batches/epoch: {len(val_loader)}')

---
## 9. Treinar!

Loop principal (~1-3h em T4 x2).

In [ ]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = torch.cuda.is_available()

# Modelo
model = PelancaNNUE()
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if num_gpus > 1:
    model = nn.DataParallel(model)
    print(f'DataParallel com {num_gpus} GPUs')
model = model.to(device)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
scaler = GradScaler() if use_amp else None

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': []}

print(f'Treino: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}')
print(f'AMP: {"ON" if use_amp else "OFF"}, GPUs: {max(num_gpus, 1)}\n')

for epoch in range(EPOCHS):
    t0 = time.time()

    # --- TRAIN ---
    model.train()
    train_sum, train_n = 0.0, 0
    for stm_in, nstm_in, target in train_loader:
        stm_in = stm_in.to(device, non_blocking=True)
        nstm_in = nstm_in.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True).unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with autocast():
                loss = nnue_loss(model(stm_in, nstm_in), target)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = nnue_loss(model(stm_in, nstm_in), target)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        train_sum += loss.item()
        train_n += 1

    train_loss = train_sum / max(train_n, 1)

    # --- VALIDATE ---
    model.eval()
    val_sum, val_n = 0.0, 0
    with torch.no_grad():
        for stm_in, nstm_in, target in val_loader:
            stm_in = stm_in.to(device, non_blocking=True)
            nstm_in = nstm_in.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True).unsqueeze(1)
            if use_amp:
                with autocast():
                    loss = nnue_loss(model(stm_in, nstm_in), target)
            else:
                loss = nnue_loss(model(stm_in, nstm_in), target)
            val_sum += loss.item()
            val_n += 1

    val_loss = val_sum / max(val_n, 1)
    scheduler.step(val_loss)

    # Log
    elapsed = time.time() - t0
    lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    m_save = model.module if hasattr(model, 'module') else model
    marker = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({'epoch': epoch, 'model_state_dict': m_save.state_dict(),
                    'val_loss': val_loss}, os.path.join(CHECKPOINT_DIR, 'best.pt'))
        marker = ' ** BEST'

    torch.save({'epoch': epoch, 'model_state_dict': m_save.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss}, os.path.join(CHECKPOINT_DIR, 'latest.pt'))

    print(f'Epoch {epoch:3d} | train={train_loss:.6f} | val={val_loss:.6f} | '
          f'lr={lr:.1e} | {elapsed:.0f}s{marker}')

    if (epoch + 1) % 10 == 0:
        p = os.path.join(CHECKPOINT_DIR, f'pelanca_e{epoch}.nnue')
        export_weights(m_save, p, verbose=False)
        print(f'  -> Exportado {p}')

print(f'\nConcluido! Melhor val_loss: {best_val_loss:.6f}')

---
## 10. Grafico de Loss

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['train_loss'][3:], label='Train')
ax2.plot(history['val_loss'][3:], label='Val')
ax2.set_xlabel('Epoch (a partir de 3)'); ax2.set_ylabel('Loss')
ax2.set_title('Loss (sem primeiras epochs)'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_loss.png', dpi=150)
plt.show()

---
## 11. Exportar Modelo Final

In [ ]:
best_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'best.pt'), map_location='cpu')
final_model = PelancaNNUE()
final_model.load_state_dict(best_ckpt['model_state_dict'])
print(f'Melhor modelo: epoch {best_ckpt["epoch"]}, val_loss={best_ckpt["val_loss"]:.6f}\n')

export_weights(final_model, NNUE_OUTPUT)

print(f'\n{"="*60}')
print(f'ARQUIVO PRONTO PARA DOWNLOAD:')
print(f'  {NNUE_OUTPUT}')
print(f'  Tamanho: {os.path.getsize(NNUE_OUTPUT):,} bytes')
print(f'')
print(f'Instrucoes:')
print(f'  1. Baixe pelanca.nnue (Output tab na barra lateral)')
print(f'  2. Copie para nn/pelanca.nnue no projeto')
print(f'  3. cargo build --release')
print(f'  4. O engine detecta o NNUE automaticamente!')
print(f'{"="*60}')

---
## 12. Teste em Posicoes Conhecidas (opcional)

In [ ]:
final_model.eval()

test_positions = [
    ('rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1', 'Posicao inicial (~0)'),
    ('rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1', 'Apos 1.e4'),
    ('8/8/8/8/8/8/4K3/R3k3 w - - 0 1', 'KR vs K (ganho brancas)'),
    ('8/8/8/8/8/8/4k3/r3K3 b - - 0 1', 'kr vs K (ganho pretas)'),
    ('4k3/8/8/8/8/8/PPPPPPPP/RNBQKBNR w KQ - 0 1', 'Material massive brancas'),
    ('rnbqkbnr/pppppppp/8/8/8/8/8/4K3 b kq - 0 1', 'Material massive pretas'),
]

print(f'{"Descricao":<35} {"Raw":>8} {"tanh":>8} {"~cp":>8}')
print('-' * 65)

for fen, desc in test_positions:
    stm, w, b = fen_to_features(fen)
    w_t = torch.from_numpy(w).unsqueeze(0)
    b_t = torch.from_numpy(b).unsqueeze(0)

    if stm == 0:
        stm_in, nstm_in = w_t, b_t
    else:
        stm_in, nstm_in = b_t, w_t

    with torch.no_grad():
        raw = final_model(stm_in, nstm_in).item()
    
    tanh_val = np.tanh(raw)
    # Converter para cp aproximado
    cp_approx = np.arctanh(np.clip(tanh_val, -0.9999, 0.9999)) * EVAL_SCALE

    print(f'{desc:<35} {raw:>+8.3f} {tanh_val:>+8.3f} {cp_approx:>+8.0f}')

print('\nValores positivos = bom para o lado a mover')